# QQQ Direction Backtest — Visual Analysis

Port of `backtest_qqq_direction.py` (no live TypeSafe calls) with rich visualizations.

**Data:** cached predictions in `backtest_predictions.jsonl` (1523 successful days, 2020–2026) + real QQQ closes via `yfinance`.
Core chart highlights **predicted UP as green background** and **predicted DOWN as red background** (`alpha=0.3`).

Run top-to-bottom: `Kernel → Restart & Run All`.

In [ ]:
import json
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
PRED_PATH = 'backtest_predictions.jsonl'
print('ready')

## 1. Load cached backtest predictions (no API calls)

In [ ]:
records = [json.loads(l) for l in open(PRED_PATH) if l.strip()]
df_all = pd.DataFrame(records)
df_all['date'] = pd.to_datetime(df_all['date'])
df_all = df_all.sort_values('date').reset_index(drop=True)

# Successful predictions only (error rows have no 'predicted')
df = df_all[df_all['predicted'].notna()].copy()
df['pred_label'] = df['predicted']  # 'UP' / 'DOWN'
df['correct'] = df['correct'].astype(bool)
df = df.set_index('date').sort_index()

n_err = len(df_all) - len(df)
acc = df['correct'].mean()
print(f"cached rows: {len(df_all)}  |  success: {len(df)}  |  errors skipped: {n_err}")
print(f"range: {df.index.min().date()} .. {df.index.max().date()}")
print(f"accuracy: {acc:.4f}  |  UP preds: {(df['pred_label']=='UP').sum()}  |  DOWN preds: {(df['pred_label']=='DOWN').sum()}")
df.head(3)

## 2. Fetch real QQQ closes & merge with predictions

In [ ]:
import yfinance as yf

start = (df.index.min() - pd.Timedelta(days=10)).strftime('%Y-%m-%d')
end = (df.index.max() + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
px = yf.download('QQQ', start=start, end=end, interval='1d', auto_adjust=True, progress=False)
# flatten MultiIndex columns ('Close','QQQ') defensively (same as .py)
if hasattr(px.columns, 'droplevel'):
    try:
        px.columns = px.columns.droplevel(1)
    except (ValueError, IndexError, KeyError, TypeError):
        pass
col = 'Close' if 'Close' in px.columns else 'Adj Close'
price = px[[col]].rename(columns={col: 'Close'}).copy()
price.index = pd.to_datetime(price.index).tz_localize(None)
price = price.dropna()
print(f"price rows: {len(price)}  col: {col}")

m = pd.merge(price, df, left_index=True, right_index=True, how='inner')
# fallback: if yfinance missed days, keep prediction order and forward-fill price
print(f"merged rows: {len(m)} / {len(df)}")
m[['Close', 'pred_label', 'actual', 'confidence', 'correct']].head(3)

## 3. ⭐ Main chart — QQQ price with prediction regimes (green=UP / red=DOWN, alpha=0.3)

In [ ]:
def add_prediction_background(ax, dates, preds, alpha=0.3):
    """Shade contiguous prediction regimes: UP=green, DOWN=red. Returns run count."""
    assert len(dates) == len(preds)
    # compress consecutive identical predictions into single spans (fast for 1500+ days)
    runs, s = [], 0
    for i in range(1, len(preds) + 1):
        if i == len(preds) or preds[i] != preds[s]:
            runs.append((dates[s], dates[i-1] + pd.Timedelta(days=1), preds[s]))
            s = i
    for d0, d1, lab in runs:
        ax.axvspan(d0, d1, color='green' if lab == 'UP' else 'red', alpha=alpha, linewidth=0)
    return len(runs)

fig, ax = plt.subplots(figsize=(14, 5))
n_runs = add_prediction_background(ax, m.index, m['pred_label'].values, alpha=0.3)
ax.plot(m.index, m['Close'], color='#1f3a93', lw=1.5, zorder=3, label='QQQ Close')
ax.set_title(f'QQQ Price with Direction Predictions  ({len(m)} days, {n_runs} regimes)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Price ($)')
handles = [mpatches.Patch(color='green', alpha=0.3, label='Predicted UP'),
           mpatches.Patch(color='red', alpha=0.3, label='Predicted DOWN'),
           plt.Line2D([], [], color='#1f3a93', label='QQQ Close')]
ax.legend(handles=handles, loc='upper left')
fig.tight_layout(); plt.show()
print(f"spans drawn: {n_runs} (alpha=0.3)")

### 3b. Zoomed view — last 6 months (regime detail + correctness dots)

In [ ]:
zoom = m.loc[m.index >= m.index.max() - pd.Timedelta(days=180)]
fig, ax = plt.subplots(figsize=(14, 4.5))
add_prediction_background(ax, zoom.index, zoom['pred_label'].values, alpha=0.3)
ax.plot(zoom.index, zoom['Close'], color='#1f3a93', lw=1.8, zorder=3)
# correctness dots: green circle = correct, black x = wrong
ok = zoom[zoom['correct']]; bad = zoom[~zoom['correct']]
ax.scatter(ok.index, ok['Close'], s=18, color='lime', edgecolors='darkgreen', linewidths=0.6, zorder=4, label=f'correct ({len(ok)})')
ax.scatter(bad.index, bad['Close'], s=22, color='black', marker='x', linewidths=1.0, zorder=4, label=f'wrong ({len(bad)})')
ax.set_title('QQQ — last 6 months: prediction regimes + correctness', fontweight='bold')
ax.set_ylabel('Price ($)'); ax.legend(loc='upper left', fontsize=9)
fig.tight_layout(); plt.show()

## 4. Strategy equity — long/short vs long/flat vs buy & hold
*Assumption (documented): `actual_change_pct` is log-return×100. Long/short takes +r on UP preds, −r on DOWN preds; long/flat takes +r on UP, 0 on DOWN.*

In [ ]:
m['logret'] = m['actual_change_pct'] / 100.0  # back to log-return decimals
m['strat_ls'] = np.where(m['pred_label'] == 'UP', m['logret'], -m['logret'])
m['strat_lf'] = np.where(m['pred_label'] == 'UP', m['logret'], 0.0)
for c in ['logret', 'strat_ls', 'strat_lf']:
    m[f'eq_{c}'] = np.exp(m[c].cumsum())  # rebased to 1.0

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(m.index, m['eq_logret'], label='Buy & Hold', color='black', lw=1.6)
ax.plot(m.index, m['eq_strat_ls'], label='Strategy long/short', color='#0a7d2c', lw=1.4)
ax.plot(m.index, m['eq_strat_lf'], label='Strategy long/flat', color='#2ca02c', lw=1.2, ls='--')
ax.fill_between(m.index, m['eq_strat_ls'], m['eq_logret'], where=m['eq_strat_ls'] >= m['eq_logret'],
                color='green', alpha=0.12, interpolate=True)
ax.fill_between(m.index, m['eq_strat_ls'], m['eq_logret'], where=m['eq_strat_ls'] < m['eq_logret'],
                color='red', alpha=0.12, interpolate=True)
ax.set_title('Equity curves (rebased 1.0) — strategy vs buy & hold', fontweight='bold')
ax.set_ylabel('Growth of $1'); ax.legend()
fig.tight_layout(); plt.show()
print(m[[c for c in m.columns if c.startswith('eq_')]].iloc[-1].round(3).to_string())

## 5. Drawdown + rolling accuracy (20-day)

In [ ]:
eq = m['eq_strat_ls']
dd = eq / eq.cummax() - 1.0
roll_acc = m['correct'].rolling(20, min_periods=5).mean()

fig, (a1, a2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True,
                             gridspec_kw={'height_ratios': [1.4, 1]})
a1.plot(m.index, eq, color='#0a7d2c', lw=1.4)
a1.fill_between(m.index, dd * eq.cummax() + eq.cummax() * 0, eq, color='red', alpha=0.15)
a1.set_title('Strategy drawdown (long/short equity vs running peak fill)', fontweight='bold')
a1.set_ylabel('Equity'); a1.text(0.01, 0.92, f"max DD: {dd.min():.1%}", transform=a1.transAxes, fontsize=11,
             bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
a2.plot(roll_acc.index, roll_acc, color='#6a1b9a', lw=1.4, label='20-day rolling accuracy')
a2.axhline(0.5, color='gray', ls='--', lw=1, label='coin flip')
a2.axhline(acc, color='red', ls=':', lw=1.2, label=f'overall {acc:.3f}')
a2.set_ylim(0, 1); a2.set_ylabel('Accuracy'); a2.legend(fontsize=9)
fig.tight_layout(); plt.show()

## 6. Confusion matrix + confidence diagnostics

In [ ]:
ct = pd.crosstab(m['pred_label'], m['actual'], rownames=['Predicted'], colnames=['Actual'])
for c in ['UP', 'DOWN']:
    for r in ['UP', 'DOWN']:
        if (r, c) not in ct.stack().index:
            pass
ct = ct.reindex(index=['UP', 'DOWN'], columns=['UP', 'DOWN'], fill_value=0)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.5))
# confusion heatmap (matplotlib only)
im = a1.imshow(ct.values, cmap='Blues')
a1.set_xticks([0, 1], ['UP', 'DOWN']); a1.set_yticks([0, 1], ['UP', 'DOWN'])
a1.set_xlabel('Actual'); a1.set_ylabel('Predicted')
a1.set_title('Confusion matrix (counts)', fontweight='bold')
for i in range(2):
    for j in range(2):
        a1.text(j, i, ct.values[i, j], ha='center', va='center', fontsize=14, fontweight='bold')
fig.colorbar(im, ax=a1, shrink=0.8)
# confidence: correct vs wrong histograms
a2.hist(m[m['correct']]['confidence'], bins=30, alpha=0.6, label='correct', color='green')
a2.hist(m[~m['correct']]['confidence'], bins=30, alpha=0.5, label='wrong', color='red')
a2.set_xlabel('Confidence'); a2.set_ylabel('Days'); a2.legend()
a2.set_title('Confidence distribution: correct vs wrong', fontweight='bold')
fig.tight_layout(); plt.show()
print(ct.to_string())
print(m.groupby(pd.cut(m['confidence'], [0, 0.6, 0.8, 1.01]))['correct'].agg(['mean', 'count']).round(4).to_string())

## 7. Return separation + yearly accuracy

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.5))
up_r = m[m['pred_label']=='UP']['actual_change_pct']
dn_r = m[m['pred_label']=='DOWN']['actual_change_pct']
a1.hist(dn_r, bins=60, alpha=0.55, color='red', label=f'DOWN pred (n={len(dn_r)})')
a1.hist(up_r, bins=30, alpha=0.65, color='green', label=f'UP pred (n={len(up_r)})')
a1.axvline(up_r.median(), color='darkgreen', ls='--', label=f'UP median {up_r.median():+.2f}%')
a1.axvline(dn_r.median(), color='darkred', ls='--', label=f'DOWN median {dn_r.median():+.2f}%')
a1.set_xlim(-6, 6); a1.set_xlabel('Realized next-day change %'); a1.set_ylabel('Days')
a1.set_title('Do UP calls earn higher realized returns?', fontweight='bold')
a1.legend(fontsize=8)
yr = m.assign(year=m.index.year).groupby('year')['correct'].agg(['mean', 'count'])
a2.bar(yr.index.astype(str), yr['mean'], color=np.where(yr['mean'] >= 0.5, 'green', 'red'), alpha=0.7)
a2.axhline(0.5, color='gray', ls='--'); a2.axhline(acc, color='black', ls=':', label=f'overall {acc:.3f}')
a2.set_ylim(0, 1); a2.set_ylabel('Accuracy'); a2.set_title('Accuracy by year', fontweight='bold')
a2.legend(fontsize=9)
for x, v, n in zip(yr.index.astype(str), yr['mean'], yr['count']):
    a2.text(x, v + 0.015, f'{v:.2f}\n(n={n})', ha='center', fontsize=7)
fig.tight_layout(); plt.show()

## 8. Monthly accuracy heatmap

In [ ]:
tmp = m.assign(y=m.index.year, mo=m.index.month)
piv = tmp.groupby(['y', 'mo'])['correct'].mean().unstack('mo')
fig, ax = plt.subplots(figsize=(12, max(3, 0.55 * len(piv))))
im = ax.imshow(piv.values, vmin=0.3, vmax=0.7, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(12), ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_yticks(range(len(piv)), piv.index.astype(str))
ax.set_title('Monthly accuracy (green ≥ 50%)', fontweight='bold')
for i in range(len(piv)):
    for j in range(12):
        v = piv.values[i, j]
        if np.isfinite(v):
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.9, label='accuracy')
fig.tight_layout(); plt.show()